# Ablation chain Y (Y1–Y6): shared encoder

Trains and evaluates ablation chain Y of the two-stage cocoa bean grading framework. The RetinaNet detector and the Swin-Tiny classifier share a single encoder, so Y1–Y4 are each retrained in full. Steps: Y1 baseline (standard anchors, linear head), Y2 differential-evolution anchors, Y3 EMDMFM fusion, Y4 EMDMFM + SAFM, Y5 XGBoost on classifier embeddings, Y6 isotonic calibration. No CLAHE preprocessing is applied. Designed to run on Google Colab with the dataset on Google Drive.


In [ ]:
!pip install torchmetrics

In [ ]:
# Ablation chain Y (Y1-Y6): detector and classifier share a single Swin-Tiny
# encoder (the same object, not a copy); no CLAHE preprocessing. This is the
# shared-encoder counterpart of chain X.
#
# Detector training updates the shared encoder weights, so Y1-Y4 each retrain
# detector and classifier from scratch. Y5 (XGBoost) and Y6 (isotonic
# calibration) do not touch the encoder and reuse the Y4 models.
#
# Requires the DE anchor configuration JSON (A4_ANCHOR_JSON) from the anchor
# optimisation step.

from google.colab import drive
drive.mount('/content/drive')

import os, json, copy, shutil, gc, warnings, pickle
import cv2
import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150})
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm import tqdm
from collections import OrderedDict
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, brier_score_loss)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import RetinaNet
from torchvision.ops import FeaturePyramidNetwork, sigmoid_focal_loss
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import timm
import xgboost as xgb
from torchmetrics.detection.mean_ap import MeanAveragePrecision

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# --- Paths ---
BASE_DRIVE_DIR  = "/content/drive/MyDrive/UMK/Datasets/MCC-02-dataset"
RESULTS_DIR     = os.path.join(BASE_DRIVE_DIR, "results_novelty_full_clahe")
CKPT_DIR        = os.path.join(BASE_DRIVE_DIR, "checkpoints_full_clahe")
ABLATION_DIR    = os.path.join(BASE_DRIVE_DIR, "ablation_results")
LOCAL_VM_DIR    = "/content"
LOCAL_SPLIT_DIR = os.path.join(LOCAL_VM_DIR, "splits")
LOCAL_IMG_DIR   = os.path.join(LOCAL_VM_DIR, "images")
DRIVE_CROP_BKUP = os.path.join(BASE_DRIVE_DIR, "cropped_splits_gt_based_backup")

for _d in [ABLATION_DIR, RESULTS_DIR, CKPT_DIR]:
    os.makedirs(_d, exist_ok=True)

A4_ANCHOR_JSON = os.path.join(CKPT_DIR, "A4_shared_swin_v2_anchor_config.json")

Y1_DET_CKPT = os.path.join(CKPT_DIR, "Y1_baseline_det.pth")
Y1_CLF_CKPT = os.path.join(CKPT_DIR, "Y1_baseline_clf.pth")
Y2_DET_CKPT = os.path.join(CKPT_DIR, "Y2_de_anchor_det.pth")
Y2_CLF_CKPT = os.path.join(CKPT_DIR, "Y2_de_anchor_clf.pth")
Y3_DET_CKPT = os.path.join(CKPT_DIR, "Y3_emdmfm_nosafm_det.pth")
Y3_CLF_CKPT = os.path.join(CKPT_DIR, "Y3_emdmfm_nosafm_clf.pth")
Y4_DET_CKPT = os.path.join(CKPT_DIR, "Y4_adaptiveswin_full_det.pth")
Y4_CLF_CKPT = os.path.join(CKPT_DIR, "Y4_adaptiveswin_full_clf.pth")
Y5_XGB_PKL  = os.path.join(CKPT_DIR, "Y5_xgb_nocal.pkl")
Y6_XGB_PKL  = os.path.join(CKPT_DIR, "Y6_xgb_calibrated_isotonic.pkl")

Y1_ID, Y2_ID, Y3_ID, Y4_ID, Y5_ID, Y6_ID = (
    "Y1_baseline", "Y2_de_anchor", "Y3_emdmfm_nosafm",
    "Y4_adaptiveswin_full", "Y5_xgb_nocal", "Y6_full_calibrated")

# Load the DE-optimised anchor configuration
if not os.path.exists(A4_ANCHOR_JSON):
    raise FileNotFoundError(f"DE anchor config not found: {A4_ANCHOR_JSON}")
with open(A4_ANCHOR_JSON) as f:
    anchor_cfg = json.load(f)
_BASE_SZ   = anchor_cfg["base_sizes"]
OPT_SCALES = anchor_cfg["opt_scales"]
OPT_RATIOS = anchor_cfg["opt_ratios"]
_expected_na_de = anchor_cfg["num_anchors_per_location"]

DE_GEN = AnchorGenerator(
    sizes=tuple(tuple(b * s for s in OPT_SCALES) for b in _BASE_SZ),
    aspect_ratios=tuple(tuple(OPT_RATIOS) for _ in _BASE_SZ))
assert DE_GEN.num_anchors_per_location()[0] == _expected_na_de
print(f"DE anchors loaded (num_anchors={_expected_na_de})")

STANDARD_SCALES, STANDARD_RATIOS = [1.0, 1.25, 1.5], [0.5, 1.0, 2.0]
STANDARD_GEN = AnchorGenerator(
    sizes=tuple(tuple(b * s for s in STANDARD_SCALES) for b in _BASE_SZ),
    aspect_ratios=tuple(tuple(STANDARD_RATIOS) for _ in _BASE_SZ))
_na_std = STANDARD_GEN.num_anchors_per_location()[0]
print(f"Standard anchors (Y1) num_anchors={_na_std}")

# --- Data loading ---
DRIVE_JSON = os.path.join(BASE_DRIVE_DIR, "annotations_coco.json")
LOCAL_JSON = os.path.join(LOCAL_VM_DIR,   "annotations_coco.json")
if not os.path.exists(LOCAL_JSON):
    shutil.copy2(DRIVE_JSON, LOCAL_JSON)
if not os.path.exists(LOCAL_IMG_DIR):
    shutil.copytree(os.path.join(BASE_DRIVE_DIR, "images"), LOCAL_IMG_DIR)
if not os.path.exists(LOCAL_SPLIT_DIR):
    shutil.copytree(os.path.join(BASE_DRIVE_DIR, "splits"), LOCAL_SPLIT_DIR)

with open(LOCAL_JSON) as _f:
    coco_data = json.load(_f)
cat_map        = {c['id']: c['name'] for c in coco_data['categories']}
sorted_cat_ids = sorted(cat_map.keys())
safe_label_map = {oid: si for si, oid in enumerate(sorted_cat_ids)}
class_names    = [cat_map[cid] for cid in sorted_cat_ids]
NUM_CLASSES    = len(class_names)
class_to_idx   = {n: i for i, n in enumerate(class_names)}
print(f"Classes: {class_names}  (N={NUM_CLASSES})")

# --- Hyperparameters (no CLAHE) ---
IMG_SIZE = 224
BATCH    = 16
DET_EP   = 25
CLF_EP   = 15

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --- Datasets ---
class DetDataset(Dataset):
    def __init__(self, coco_json, root, filter_dir=None):
        self.root = root
        valid = None
        if filter_dir:
            valid = set()
            for cls in os.listdir(filter_dir):
                d = os.path.join(filter_dir, cls)
                if os.path.isdir(d):
                    for fname in os.listdir(d): valid.add(fname)
        self.images = [img for img in coco_json['images']
                       if valid is None or img['file_name'].split('/')[-1] in valid]
        self.anns = {}
        for ann in coco_json['annotations']:
            self.anns.setdefault(ann['image_id'], []).append(ann)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        info = self.images[idx]
        img  = Image.open(os.path.join(self.root, info['file_name']))
        img  = ImageOps.exif_transpose(img).convert("RGB")
        img  = transforms.ToTensor()(img)
        anns = self.anns.get(info['id'], [])
        boxes  = [[a['bbox'][0], a['bbox'][1],
                   a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]] for a in anns]
        labels = [safe_label_map[a['category_id']] + 1 for a in anns]
        if boxes:
            t = {"boxes":  torch.as_tensor(boxes,  dtype=torch.float32),
                 "labels": torch.as_tensor(labels, dtype=torch.int64)}
        else:
            t = {"boxes":  torch.empty((0, 4), dtype=torch.float32),
                 "labels": torch.empty((0,),   dtype=torch.int64)}
        return img, t


class ClfDataset(Dataset):
    def __init__(self, df, tf=None):
        self.df = df; self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path'])
        img = ImageOps.exif_transpose(img).convert('RGB')
        if self.tf: img = self.tf(img)
        return img, int(row['label']), row['image_path']


def collate_fn(b): return tuple(zip(*b))


def make_loaders(crop_dir):
    def _df(split):
        data = []
        for cls in class_names:
            d = os.path.join(crop_dir, split, cls)
            if not os.path.exists(d): continue
            for fname in os.listdir(d):
                if fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                    data.append({'image_path': os.path.join(d, fname),
                                 'label': class_to_idx[cls]})
        return pd.DataFrame(data).reset_index(drop=True)
    tr = _df('train'); va = _df('val'); te = _df('test')
    print(f"  Dataset: train={len(tr)}, val={len(va)}, test={len(te)}")
    mkl = lambda df, sh, tf: DataLoader(
        ClfDataset(df, tf), batch_size=BATCH, shuffle=sh,
        pin_memory=True, num_workers=2, persistent_workers=True)
    return mkl(tr, True, train_tf), mkl(va, False, val_tf), mkl(te, False, val_tf), tr, va, te

# --- GT crops (copied from Drive backup if available, so the test set is identical across chains) ---
SHARED_CROP_DIR = os.path.join(LOCAL_VM_DIR, "crops_shared_gt_y1y6")

def prepare_shared_gt_crop():
    check = os.path.join(SHARED_CROP_DIR, 'test', class_names[0])
    if os.path.exists(check) and len(os.listdir(check)) > 0:
        print(f"  GT crops already present, skipping"); return
    drive_check = os.path.join(DRIVE_CROP_BKUP, 'test', class_names[0])
    if os.path.exists(drive_check) and len(os.listdir(drive_check)) > 0:
        print("  Copying GT crops from Drive backup...")
        shutil.copytree(DRIVE_CROP_BKUP, SHARED_CROP_DIR)
        print("  Copy finished")
        return
    print("  No backup found, generating GT crops...")
    for sp in ['train', 'val', 'test']:
        for cls in class_names:
            os.makedirs(os.path.join(SHARED_CROP_DIR, sp, cls), exist_ok=True)
    fname_map = {}
    for sp in ['train', 'val', 'test']:
        for cls in class_names:
            d = os.path.join(LOCAL_SPLIT_DIR, sp, cls)
            if not os.path.exists(d): continue
            for fname in os.listdir(d):
                if fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                    fname_map[fname] = (sp, cls)
    img_anns = {}
    for ann in coco_data['annotations']:
        img_anns.setdefault(ann['image_id'], []).append(ann)
    n = 0
    for info in tqdm(coco_data['images'], desc="  GT Crop"):
        fname = info['file_name'].split('/')[-1]
        if fname not in fname_map: continue
        sp, cls = fname_map[fname]
        src = os.path.join(LOCAL_SPLIT_DIR, sp, cls, fname)
        if not os.path.exists(src): continue
        anns = img_anns.get(info['id'], [])
        if not anns: continue
        try:
            orig = Image.open(src); orig = ImageOps.exif_transpose(orig).convert("RGB")
            iw, ih = orig.size
            for ci, ann in enumerate(anns):
                x, y, w, h = ann['bbox']
                x0, y0 = max(0, int(x)), max(0, int(y))
                x1, y1 = min(iw, int(x+w)), min(ih, int(y+h))
                if (x1-x0) < 10 or (y1-y0) < 10: continue
                orig.crop((x0, y0, x1, y1)).save(
                    os.path.join(SHARED_CROP_DIR, sp, cls,
                                 f"crop_{fname.split('.')[0]}_{ci}.jpg"))
                n += 1
        except Exception as e:
            print(f"  [WARN] {fname}: {e}")
    print(f"  Generated {n} crops")

# --- Training and evaluation helpers ---
def train_detector(det_model, ckpt_path, extra_meta=None):
    det_ds = DetDataset(coco_data, LOCAL_VM_DIR)
    det_ld = DataLoader(det_ds, batch_size=64, shuffle=True, collate_fn=collate_fn,
                        num_workers=2, pin_memory=True, persistent_workers=True)
    opt = optim.SGD(det_model.parameters(), lr=0.002, momentum=0.9, weight_decay=5e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=DET_EP, eta_min=1e-5)
    start = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        det_model.load_state_dict(ck['model_state']); opt.load_state_dict(ck['optimizer_state'])
        if 'scheduler_state' in ck: sch.load_state_dict(ck['scheduler_state'])
        start = ck['epoch'] + 1
        print(f"  Resuming detector from epoch {start}/{DET_EP}")
    if start >= DET_EP:
        print(f"  Detector already trained, skipping."); return det_model
    det_model.train()
    for ep in range(start, DET_EP):
        ls = 0.0; nok = 0; nnan = 0
        for imgs, tgts in tqdm(det_ld, desc=f"  Det {ep+1}/{DET_EP}"):
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            loss = sum(det_model(imgs, tgts).values())
            if not torch.isfinite(loss): nnan += 1; opt.zero_grad(); continue
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(det_model.parameters(), 10.0)
            opt.step(); ls += loss.item(); nok += 1
        sch.step(); avg = ls / max(1, nok)
        print(f"  Ep {ep+1:02d}/{DET_EP} Loss={avg:.4f} LR={opt.param_groups[0]['lr']:.2e} NaN={nnan}")
        save_payload = {'epoch': ep, 'model_state': det_model.state_dict(),
                        'optimizer_state': opt.state_dict(),
                        'scheduler_state': sch.state_dict(), 'loss': avg}
        if extra_meta: save_payload.update(extra_meta)
        torch.save(save_payload, ckpt_path)
    return det_model

def eval_map(det_model):
    test_dir = os.path.join(LOCAL_SPLIT_DIR, 'test')
    te_ds    = DetDataset(coco_data, LOCAL_VM_DIR, filter_dir=test_dir)
    te_ld    = DataLoader(te_ds, batch_size=16, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    det_model.eval()
    with torch.no_grad():
        for imgs, tgts in tqdm(te_ld, desc="  mAP eval"):
            imgs = [i.to(DEVICE) for i in imgs]; outs = det_model(imgs)
            metric.update(
                [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(),
                  "labels": o["labels"].cpu()} for o in outs],
                [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in tgts])
    r = metric.compute()
    m5095, m50, m75 = float(r['map'].item()), float(r['map_50'].item()), float(r['map_75'].item())
    print(f"  mAP@50={m50:.4f}  mAP@75={m75:.4f}  mAP@50:95={m5095:.4f}")
    return m5095, m50, m75

def train_classifier(clf_model, tr_ld, va_ld, ckpt_path):
    all_labels = [int(r['label']) for _, r in tr_ld.dataset.df.iterrows()]
    try:
        cw = torch.tensor(
            compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels),
            dtype=torch.float).to(DEVICE)
    except Exception: cw = None
    crit   = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt    = optim.AdamW(clf_model.parameters(), lr=1e-4, weight_decay=1e-4)
    sch    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLF_EP, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None
    start  = 0; best_acc = 0.0; best_wts = copy.deepcopy(clf_model.state_dict())
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        clf_model.load_state_dict(ck['model_state']); opt.load_state_dict(ck['optimizer_state'])
        if 'scheduler_state' in ck: sch.load_state_dict(ck['scheduler_state'])
        start    = ck['epoch'] + 1
        best_acc = ck.get('best_val_acc', 0.0)
        best_wts = copy.deepcopy(clf_model.state_dict())
        print(f"  Resuming classifier from epoch {start}/{CLF_EP} (best={best_acc:.4f})")
    if start >= CLF_EP:
        print(f"  Classifier already trained, skipping.")
        clf_model.load_state_dict(best_wts); return clf_model, best_acc
    for ep in range(start, CLF_EP):
        clf_model.train(); tot = 0; corr = 0; el = 0.0
        for imgs, lbls, _ in tqdm(tr_ld, desc=f"  Clf {ep+1}/{CLF_EP}"):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE); opt.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'): out = clf_model(imgs); loss = crit(out, lbls)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(clf_model.parameters(), 2.0)
                scaler.step(opt); scaler.update()
            else:
                out = clf_model(imgs); loss = crit(out, lbls); loss.backward()
                torch.nn.utils.clip_grad_norm_(clf_model.parameters(), 2.0); opt.step()
            el += loss.item()*imgs.size(0); corr += (out.argmax(1)==lbls).sum(); tot += imgs.size(0)
        sch.step()
        clf_model.eval(); vc = 0; vt = 0
        with torch.no_grad():
            for imgs, lbls, _ in va_ld:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                vc += (clf_model(imgs).argmax(1) == lbls).sum(); vt += imgs.size(0)
        va = float(vc/vt) if vt > 0 else 0.0
        print(f"  Ep {ep+1:02d}/{CLF_EP}  Train={float(corr/tot):.4f}  Val={va:.4f}")
        if va > best_acc: best_acc = va; best_wts = copy.deepcopy(clf_model.state_dict())
        torch.save({'epoch': ep, 'model_state': clf_model.state_dict(),
                    'optimizer_state': opt.state_dict(), 'scheduler_state': sch.state_dict(),
                    'best_val_acc': best_acc}, ckpt_path)
    clf_model.load_state_dict(best_wts)
    print(f"\n  Training finished. Best val acc = {best_acc:.4f}")
    return clf_model, best_acc

def compute_ece(y_true, y_proba, n_bins=15):
    conf = y_proba.max(axis=1); pred = y_proba.argmax(axis=1)
    acc  = (pred == y_true).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece = 0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1])
        if m.sum() == 0: continue
        ece += (m.sum()/len(y_true)) * abs(acc[m].mean() - conf[m].mean())
    return ece

def eval_classifier(clf_model, te_ld, ablation_id=""):
    clf_model.eval(); yt = []; yp = []
    with torch.no_grad():
        for imgs, lbls, _ in tqdm(te_ld, desc=f"  Clf eval [{ablation_id}]"):
            yt.extend(lbls.numpy())
            yp.extend(clf_model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
    yt = np.array(yt); yp = np.array(yp)
    acc      = float(accuracy_score(yt, yp))
    f1_macro = float(f1_score(yt, yp, average='macro'))
    report_dict = classification_report(
        yt, yp, target_names=class_names, digits=4, output_dict=True)
    per_class = {}
    for cls in class_names:
        d = report_dict[cls]; support = int(d['support']); recall = float(d['recall'])
        missed = support - int(round(recall * support))
        per_class[cls] = {"precision": round(float(d['precision']), 4),
                          "recall": round(recall, 4), "f1": round(float(d['f1-score']), 4),
                          "support": support, "missed": missed}
    print(f"\n  [{ablation_id}] Acc={acc:.4f}  F1-macro={f1_macro:.4f}")
    print(f"  {'Class':<15} {'P':>8} {'R':>8} {'F1':>8} {'Miss':>6}")
    for cls in class_names:
        d = per_class[cls]
        print(f"  {cls:<15} {d['precision']:>8.4f} {d['recall']:>8.4f} "
              f"{d['f1']:>8.4f} {d['missed']:>6}")
    cm = confusion_matrix(yt, yp)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix -- {ablation_id}')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    cm_path = os.path.join(ABLATION_DIR, f"confusion_matrix_{ablation_id}.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"  Confusion matrix: {cm_path}")
    return acc, f1_macro, per_class

def eval_from_labels(y_true, y_pred, ablation_id):
    acc = float(accuracy_score(y_true, y_pred))
    f1m = float(f1_score(y_true, y_pred, average='macro'))
    rpt = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
    pc = {}
    for cls in class_names:
        d = rpt[cls]; s = int(d['support']); r = float(d['recall'])
        pc[cls] = {"precision": round(float(d['precision']),4), "recall": round(r,4),
                   "f1": round(float(d['f1-score']),4), "support": s,
                   "missed": s - int(round(r*s))}
    print(f"\n  [{ablation_id}] Acc={acc:.4f}  F1-macro={f1m:.4f}")
    for cls in class_names:
        d = pc[cls]
        print(f"  {cls:<15} P={d['precision']:.4f} R={d['recall']:.4f} F1={d['f1']:.4f} Miss={d['missed']}")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6)); sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
        xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix -- {ablation_id}'); plt.tight_layout()
    cmp_path = os.path.join(ABLATION_DIR, f"confusion_matrix_{ablation_id}.png")
    plt.savefig(cmp_path, dpi=300, bbox_inches='tight'); plt.show()
    print(f"  Confusion matrix: {cmp_path}")
    return acc, f1m, pc

def save_result(ablation_id, metrics):
    path = os.path.join(ABLATION_DIR, f"{ablation_id}.json")
    with open(path, 'w') as f:
        json.dump({'id': ablation_id, **metrics}, f, indent=2)
    print(f"\n{'='*65}\n{ablation_id}  ->  {path}")
    for k, v in metrics.items():
        if k == 'per_class' or v is None: continue
        print(f"    {k:<20}: {v:.4f}" if isinstance(v, float) else f"    {k:<20}: {v}")
    print('=' * 65)

# --- Model definitions ---
class TunedRetinaNetHead(RetinaNetClassificationHead):
    """RetinaNet classification head with configurable focal-loss alpha/gamma."""
    def __init__(self, *args, focal_alpha=0.5, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_alpha = focal_alpha; self.focal_gamma = focal_gamma
    def compute_loss(self, targets, head_outputs, matched_idxs):
        losses = []; cls_logits = head_outputs["cls_logits"]
        for tgt, cl_p, mi_p in zip(targets, cls_logits, matched_idxs):
            fg = torch.where(mi_p >= 0)[0]; nf = fg.numel()
            gt_ = torch.zeros_like(cl_p)
            gt_[fg, tgt["labels"][mi_p[fg]]] = 1.0
            vi = mi_p != -2
            losses.append(sigmoid_focal_loss(cl_p[vi], gt_[vi],
                    alpha=self.focal_alpha, gamma=self.focal_gamma,
                    reduction="sum") / max(1, nf))
        return sum(losses) / len(targets)


class BasicSwinCLF(nn.Module):
    """Y1/Y2: Swin-Tiny with a linear head (encoder shared with the detector)."""
    def __init__(self, num_classes):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224',
            pretrained=True, features_only=True, out_indices=(0, 1, 2))
        self.gap        = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(384, num_classes)
    def _bchw(self, x):
        if x.dim() == 4:
            return x if x.shape[1] in [96, 192, 384, 768] else x.permute(0, 3, 1, 2)
        B, L, C = x.shape; H = int(np.sqrt(L))
        return x.permute(0, 2, 1).view(B, C, H, H)
    def forward(self, x):
        return self.classifier(self.gap(self._bchw(self.swin(x)[2])).flatten(1))


class SwinCNN_NoSAFM(nn.Module):
    """Y3: EMDMFM without SAFM (shared encoder)."""
    def __init__(self, num_classes):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224',
            pretrained=True, features_only=True, out_indices=(0, 1, 2))
        self.em0 = nn.Conv2d(96, 128, 1); self.em1 = nn.Conv2d(192, 128, 1)
        self.em2 = nn.Conv2d(384, 128, 1)
        self.dm2 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm1 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm0 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.fm0 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm1 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm2 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_agg     = nn.Conv2d(384, 256, 1)
        self.gap        = nn.AdaptiveAvgPool2d(1)
        self.fc_latent  = nn.Linear(128 * 3 + 256, 384)
        self.classifier = nn.Linear(384, num_classes)
    def _bchw(self, x):
        if x.dim() == 4:
            return x if x.shape[1] in [96, 192, 384, 768] else x.permute(0, 3, 1, 2)
        B, L, C = x.shape; H = int(np.sqrt(L))
        return x.permute(0, 2, 1).view(B, C, H, H)
    def get_latent_features(self, x):
        feats = self.swin(x)
        e0 = self.em0(self._bchw(feats[0])); e1 = self.em1(self._bchw(feats[1]))
        e2 = self.em2(self._bchw(feats[2]))
        d2 = self.dm2(e2)
        d1 = self.dm1(F.interpolate(d2, size=e1.shape[2:], mode='bilinear', align_corners=False) + e1)
        d0 = self.dm0(F.interpolate(d1, size=e0.shape[2:], mode='bilinear', align_corners=False) + e0)
        f2 = self.fm2(torch.cat([e2, d2], 1)); f1 = self.fm1(torch.cat([e1, d1], 1))
        f0 = self.fm0(torch.cat([e0, d0], 1))
        fagg = self.fm_agg(torch.cat([
            f0, F.interpolate(f1, size=f0.shape[2:], mode='bilinear', align_corners=False),
            F.interpolate(f2, size=f0.shape[2:], mode='bilinear', align_corners=False)], 1))
        latent = torch.cat([self.gap(f0).flatten(1), self.gap(f1).flatten(1),
                            self.gap(f2).flatten(1), self.gap(fagg).flatten(1)], dim=1)
        return F.relu(self.fc_latent(latent))
    def forward(self, x): return self.classifier(self.get_latent_features(x))


class AdaptiveSwinCNN_EMDMFM(nn.Module):
    """Y4/Y5/Y6: EMDMFM + SAFM (shared encoder)."""
    def __init__(self, num_classes=4):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224',
            pretrained=True, features_only=True, out_indices=(0, 1, 2))
        self.em_proj0 = nn.Conv2d(96, 128, 1); self.em_proj1 = nn.Conv2d(192, 128, 1)
        self.em_proj2 = nn.Conv2d(384, 128, 1)
        self.dm_conv2 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm_conv1 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm_conv0 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.fm_conv0 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_conv1 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_conv2 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_agg   = nn.Conv2d(384, 256, 1)
        self.gap            = nn.AdaptiveAvgPool2d(1)
        self.safm_attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=True)
        self.safm_norm      = nn.LayerNorm(128)
        self.safm_dropout   = nn.Dropout(0.1)
        self.fc_latent  = nn.Linear(128 * 3 + 256, 384)
        self.classifier = nn.Linear(384, num_classes)
    def _bchw(self, x):
        if x.dim() == 4:
            return x if x.shape[1] in [96, 192, 384, 768] else x.permute(0, 3, 1, 2)
        B, L, C = x.shape; H = int(np.sqrt(L))
        return x.permute(0, 2, 1).view(B, C, H, H)
    def get_latent_features(self, x):
        features = self.swin(x)
        e0 = self.em_proj0(self._bchw(features[0])); e1 = self.em_proj1(self._bchw(features[1]))
        e2 = self.em_proj2(self._bchw(features[2]))
        d2 = self.dm_conv2(e2)
        d1 = self.dm_conv1(F.interpolate(d2, size=e1.shape[2:], mode='bilinear', align_corners=False) + e1)
        d0 = self.dm_conv0(F.interpolate(d1, size=e0.shape[2:], mode='bilinear', align_corners=False) + e0)
        f2 = self.fm_conv2(torch.cat([e2, d2], dim=1)); f1 = self.fm_conv1(torch.cat([e1, d1], dim=1))
        f0 = self.fm_conv0(torch.cat([e0, d0], dim=1))
        f_agg = self.fm_agg(torch.cat([
            f0, F.interpolate(f1, size=f0.shape[2:], mode='bilinear', align_corners=False),
            F.interpolate(f2, size=f0.shape[2:], mode='bilinear', align_corners=False)], dim=1))
        f_agg_pool = self.gap(f_agg).flatten(1)
        tokens = torch.cat([self.gap(f).flatten(1).unsqueeze(1) for f in [f0, f1, f2]], dim=1)
        attn_out, _ = self.safm_attention(tokens, tokens, tokens)
        latent = torch.cat([self.safm_norm(tokens + self.safm_dropout(attn_out)).flatten(1),
                            f_agg_pool], dim=1)
        return F.relu(self.fc_latent(latent))
    def forward(self, x): return self.classifier(self.get_latent_features(x))


class SharedSwinBackboneWrapper(nn.Module):
    """Detector backbone that shares the classifier's Swin-Tiny encoder
    (self.swin = clf_model.swin: the same object, not a copy) + FPN."""
    def __init__(self, clf_model):
        super().__init__()
        self.swin         = clf_model.swin
        self.out_channels = 256
        self.fpn = FeaturePyramidNetwork([96, 192, 384], self.out_channels)
        self.ds1 = nn.Conv2d(256, 256, 3, stride=2, padding=1)
        self.ds2 = nn.Conv2d(256, 256, 3, stride=2, padding=1)
    def _bchw(self, x):
        if x.dim() == 4:
            return x if x.shape[1] in [96, 192, 384, 768] else x.permute(0, 3, 1, 2)
        B, L, C = x.shape; H = int(np.sqrt(L))
        return x.permute(0, 2, 1).view(B, C, H, H)
    def forward(self, x):
        sf = self.swin(x)
        xd = OrderedDict([(str(i), self._bchw(sf[i])) for i in range(3)])
        fp = self.fpn(xd)
        out = OrderedDict([("0", fp["0"]), ("1", fp["1"]), ("2", fp["2"])])
        out["3"] = self.ds1(out["2"]); out["4"] = self.ds2(out["3"])
        return out


def build_detector(clf_model, anchor_gen):
    det = RetinaNet(backbone=SharedSwinBackboneWrapper(clf_model), num_classes=NUM_CLASSES+1,
        anchor_generator=anchor_gen, min_size=224, max_size=224,
        score_thresh=0.05, nms_thresh=0.5, fg_iou_thresh=0.4, bg_iou_thresh=0.3).to(DEVICE)
    _ic = det.backbone.out_channels
    _na = det.anchor_generator.num_anchors_per_location()[0]
    det.head.classification_head = TunedRetinaNetHead(
        _ic, _na, NUM_CLASSES+1, focal_alpha=0.5, focal_gamma=2.0).to(DEVICE)
    return det

# --- Run Y1 -> Y6 ---
prepare_shared_gt_crop()
tr_ld, va_ld, te_ld, tr_df, va_df, te_df = make_loaders(SHARED_CROP_DIR)

# --- Y1: shared encoder, standard anchors, linear classifier ---
print(f"\n{'='*70}\n  {Y1_ID}: Shared Backbone + Standard Anchor + Linear (no CLAHE)\n{'='*70}")
clf_y1 = BasicSwinCLF(NUM_CLASSES).to(DEVICE)
det_y1 = build_detector(clf_y1, STANDARD_GEN)
train_detector(det_y1, Y1_DET_CKPT, extra_meta={"ablation_id": Y1_ID, "anchor_type": "standard"})
map_5095_y1, map_50_y1, map_75_y1 = eval_map(det_y1)
clf_y1, best_acc_y1 = train_classifier(clf_y1, tr_ld, va_ld, Y1_CLF_CKPT)
acc_y1, f1_y1, pc_y1 = eval_classifier(clf_y1, te_ld, Y1_ID)
save_result(Y1_ID, {
    "map_5095": map_5095_y1, "map_50": map_50_y1, "map_75": map_75_y1,
    "accuracy": acc_y1, "f1_macro": f1_y1, "ece": None, "brier": None,
    "clf_conf_std": None, "best_val_acc": best_acc_y1, "per_class": pc_y1,
    "backbone": "shared", "use_clahe": False, "anchor_type": "standard",
    "head_type": "TunedRetinaNetClassificationHead",
    "note": ('Baseline: shared Swin-Tiny backbone + linear head, standard anchors, no CLAHE. Paired with X1.'),
})
gc.collect(); torch.cuda.empty_cache()

# --- Y2: DE anchors; full retrain (shared encoder prevents reuse) ---
print(f"\n{'='*70}\n  {Y2_ID}: Shared Backbone + DE Anchor -- full retrain\n{'='*70}")
clf_y2 = BasicSwinCLF(NUM_CLASSES).to(DEVICE)
det_y2 = build_detector(clf_y2, DE_GEN)
train_detector(det_y2, Y2_DET_CKPT, extra_meta={
    "ablation_id": Y2_ID, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON})
map_5095_y2, map_50_y2, map_75_y2 = eval_map(det_y2)
clf_y2, best_acc_y2 = train_classifier(clf_y2, tr_ld, va_ld, Y2_CLF_CKPT)
acc_y2, f1_y2, pc_y2 = eval_classifier(clf_y2, te_ld, Y2_ID)
save_result(Y2_ID, {
    "map_5095": map_5095_y2, "map_50": map_50_y2, "map_75": map_75_y2,
    "accuracy": acc_y2, "f1_macro": f1_y2, "ece": None, "brier": None,
    "clf_conf_std": None, "best_val_acc": best_acc_y2, "per_class": pc_y2,
    "backbone": "shared", "use_clahe": False, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON,
    "head_type": "TunedRetinaNetClassificationHead",
    "note": ('+ DE anchors. Fully retrained (not reused): with a shared backbone the Swin weights of Y1 and Y2 differ.'),
})
gc.collect(); torch.cuda.empty_cache()

# --- Y3: EMDMFM without SAFM; full retrain ---
print(f"\n{'='*70}\n  {Y3_ID}: Shared Backbone + EMDMFM (noSAFM) -- full retrain\n{'='*70}")
clf_y3 = SwinCNN_NoSAFM(NUM_CLASSES).to(DEVICE)
det_y3 = build_detector(clf_y3, DE_GEN)
train_detector(det_y3, Y3_DET_CKPT, extra_meta={
    "ablation_id": Y3_ID, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON})
map_5095_y3, map_50_y3, map_75_y3 = eval_map(det_y3)
clf_y3, best_acc_y3 = train_classifier(clf_y3, tr_ld, va_ld, Y3_CLF_CKPT)
acc_y3, f1_y3, pc_y3 = eval_classifier(clf_y3, te_ld, Y3_ID)
save_result(Y3_ID, {
    "map_5095": map_5095_y3, "map_50": map_50_y3, "map_75": map_75_y3,
    "accuracy": acc_y3, "f1_macro": f1_y3, "ece": None, "brier": None,
    "clf_conf_std": None, "best_val_acc": best_acc_y3, "per_class": pc_y3,
    "backbone": "shared", "use_clahe": False, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON,
    "head_type": "TunedRetinaNetClassificationHead",
    "note": ('+ EMDMFM without SAFM. Fully retrained. Paired with X3 (decoupled) and A5 (shared + CLAHE).'),
})
gc.collect(); torch.cuda.empty_cache()

# --- Y4: EMDMFM + SAFM; full retrain ---
print(f"\n{'='*70}\n  {Y4_ID}: Shared Backbone + EMDMFM+SAFM -- full retrain\n{'='*70}")
clf_y4 = AdaptiveSwinCNN_EMDMFM(NUM_CLASSES).to(DEVICE)
det_y4 = build_detector(clf_y4, DE_GEN)
train_detector(det_y4, Y4_DET_CKPT, extra_meta={
    "ablation_id": Y4_ID, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON})
map_5095_y4, map_50_y4, map_75_y4 = eval_map(det_y4)
clf_y4, best_acc_y4 = train_classifier(clf_y4, tr_ld, va_ld, Y4_CLF_CKPT)
acc_y4, f1_y4, pc_y4 = eval_classifier(clf_y4, te_ld, Y4_ID)
save_result(Y4_ID, {
    "map_5095": map_5095_y4, "map_50": map_50_y4, "map_75": map_75_y4,
    "accuracy": acc_y4, "f1_macro": f1_y4, "ece": None, "brier": None,
    "clf_conf_std": None, "best_val_acc": best_acc_y4, "per_class": pc_y4,
    "backbone": "shared", "use_clahe": False, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON,
    "head_type": "TunedRetinaNetClassificationHead",
    "note": ('+ SAFM (full AdaptiveSwin-CNN, shared backbone). Fully retrained. Paired with X4 (decoupled) and A6 (shared + CLAHE).'),
})
gc.collect(); torch.cuda.empty_cache()

# --- Y5: + XGBoost (uncalibrated); reuses the Y4 classifier (encoder untouched) ---
print(f"\n{'='*70}\n  {Y5_ID}: + XGBoost (raw) -- classifier reuse Y4\n{'='*70}")
def extract_hybrid_embeddings(clf_model, loader, desc):
    embeddings, labels_list = [], []
    clf_model.eval()
    with torch.no_grad():
        for imgs, lbls, _ in tqdm(loader, desc=desc):
            feat = clf_model.get_latent_features(imgs.to(DEVICE))
            embeddings.append(feat.cpu().numpy()); labels_list.extend(lbls.numpy())
    return np.vstack(embeddings), np.array(labels_list)

X_train, y_train = extract_hybrid_embeddings(clf_y4, tr_ld, "Features Train")
X_val,   y_val   = extract_hybrid_embeddings(clf_y4, va_ld, "Features Val")
X_test,  y_test  = extract_hybrid_embeddings(clf_y4, te_ld, "Features Test")

xgb_y5 = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    objective='multi:softprob', num_class=NUM_CLASSES, tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu', random_state=42)
xgb_y5.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
proba_raw = xgb_y5.predict_proba(X_test)
y_pred_raw = proba_raw.argmax(axis=1)
ece_y5 = compute_ece(y_test, proba_raw)
brier_y5 = np.mean([brier_score_loss(y_test == c, proba_raw[:, c]) for c in range(NUM_CLASSES)])
conf_std_y5 = proba_raw.max(axis=1).std()

with open(Y5_XGB_PKL, 'wb') as f: pickle.dump(xgb_y5, f)
print(f"  ECE (raw)={ece_y5:.4f}  Brier={brier_y5:.4f}")

acc_y5, f1_y5, pc_y5 = eval_from_labels(y_test, y_pred_raw, Y5_ID)
save_result(Y5_ID, {
    "map_5095": map_5095_y4, "map_50": map_50_y4, "map_75": map_75_y4,  # reuse Y4
    "accuracy": acc_y5, "f1_macro": f1_y5, "ece": float(ece_y5),
    "brier": float(brier_y5), "clf_conf_std": float(conf_std_y5),
    "best_val_acc": best_acc_y4, "per_class": pc_y5,
    "backbone": "shared", "use_clahe": False, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON,
    "built_on_classifier": Y4_CLF_CKPT,
    "note": '+ XGBoost (uncalibrated). Classifier identical to Y4 (reused; XGBoost does not change the backbone).',
})
gc.collect(); torch.cuda.empty_cache()

# --- Y6: + isotonic calibration of the Y5 XGBoost model ---
print(f"\n{'='*70}\n  {Y6_ID}: + Isotonic Calibration -- XGBoost reuse Y5\n{'='*70}")
xgb_y6_cal = CalibratedClassifierCV(xgb_y5, method='isotonic', cv='prefit')
xgb_y6_cal.fit(X_val, y_val)
proba_cal = xgb_y6_cal.predict_proba(X_test)
y_pred_cal = proba_cal.argmax(axis=1)
ece_y6 = compute_ece(y_test, proba_cal)
brier_y6 = np.mean([brier_score_loss(y_test == c, proba_cal[:, c]) for c in range(NUM_CLASSES)])
conf_std_y6 = proba_cal.max(axis=1).std()

with open(Y6_XGB_PKL, 'wb') as f: pickle.dump(xgb_y6_cal, f)
print(f"  ECE: {ece_y5:.4f} -> {ece_y6:.4f} (reduction {(ece_y5-ece_y6)/ece_y5*100:.1f}%)")

acc_y6, f1_y6, pc_y6 = eval_from_labels(y_test, y_pred_cal, Y6_ID)
save_result(Y6_ID, {
    "map_5095": map_5095_y4, "map_50": map_50_y4, "map_75": map_75_y4,  # reuse Y4
    "accuracy": acc_y6, "f1_macro": f1_y6, "ece": float(ece_y6),
    "brier": float(brier_y6), "clf_conf_std": float(conf_std_y6),
    "best_val_acc": best_acc_y4, "per_class": pc_y6,
    "backbone": "shared", "use_clahe": False, "anchor_type": "de_optimized",
    "anchor_config_source": A4_ANCHOR_JSON,
    "built_on_xgb": Y5_XGB_PKL,
    "ece_before_calibration": float(ece_y5), "brier_before_calibration": float(brier_y5),
    "note": ('+ Isotonic calibration. XGBoost identical to Y5. Full pipeline with a shared backbone and no CLAHE; compare with X6 and A8.'),
})
gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*70)
print("  Done: Y1 -> Y2 -> Y3 -> Y4 -> Y5 -> Y6")
print("="*70)
print(f"""
  Test accuracy (GT-crop test set):
    Y1={acc_y1:.4f}  Y2={acc_y2:.4f}  Y3={acc_y3:.4f}  Y4={acc_y4:.4f}
    Y5={acc_y5:.4f}  Y6={acc_y6:.4f}
""")